# 8-Model Recommendation Comparison

This notebook compares 8 different recommendation approaches on the link prediction task.

## Models Compared

| # | Model | Description |
|---|-------|-------------|
| 1 | BGE-M3 Only | Cosine similarity on BGE-M3 embeddings |
| 2 | Node2Vec Only (corrected) | Power-law corrected Node2Vec |
| 3 | BGE-M3 + Node2Vec (corrected) | Combined with power-law correction |
| 4 | GCN Only | GCN-learned embeddings (no text) |
| 5 | Node2Vec Only (uncorrected) | Raw Node2Vec without correction |
| 6 | BGE-M3 + Node2Vec (uncorrected) | Combined without correction |
| 7 | GCN + BGE-M3 Combined | Rank fusion: GCN + BGE-M3 ranks |
| 8 | GraphSAGE Only | GraphSAGE-learned embeddings |

## Metrics
- Hit@1, Hit@10, Hit@50, Hit@100
- MRR (Mean Reciprocal Rank)
- NDCG
- Recall
- MAP



In [1]:
import os, pickle, json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT = r'C:\programming\github-repos\graph-ending'
CACHE_DIR = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'eval-2')
EVAL_CACHE = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'eval-1')
GCN_CACHE = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'gcn')
SAGE_CACHE = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'graphSAGE')
os.makedirs(CACHE_DIR, exist_ok=True)

DATA_PATH = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'data', 'data-embeddings.json')
BGE_PATH = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cap-embeddings', 'BAAI_bge-m3', 'master_embeddings.parquet')
N2V_PATH = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cap-embeddings', 'node2vec', 'master_embeddings.parquet')

print('Setup complete.')

Setup complete.


## Step 1 - Load Train/Test Split and Graph Degrees


In [2]:
# Load train/test split
with open(os.path.join(EVAL_CACHE, 'train_test_split.pkl'), 'rb') as f:
    split_data = pickle.load(f)
E_test = split_data['E_test']
eval_nodes = set(split_data['eval_nodes'])
G_train = split_data['G_train']
print(f'Eval nodes: {len(eval_nodes)}, Test edges: {len(E_test)}')

# Get alpha for power-law correction
with open(os.path.join(EVAL_CACHE, 'alpha_train.pkl'), 'rb') as f:
    alpha_data = pickle.load(f)
alpha = alpha_data['alpha']
print(f'Power-law alpha: {alpha}')

# Build undirected graph and get degrees
G_train_und = G_train.to_undirected()
degrees = dict(G_train_und.degree())
print(f'Graph: {G_train_und.number_of_nodes()} nodes, {G_train_und.number_of_edges()} edges')

Eval nodes: 11256, Test edges: 63106
Power-law alpha: 2.9763366263368836
Graph: 11701 nodes, 181315 edges


## Step 2 - Load All Embeddings


In [3]:
# 1. BGE-M3
df_bge = pd.read_parquet(BGE_PATH)
bge_ids = df_bge['id'].astype(int).values
bge_emb = np.stack(df_bge['embedding'].values)
bge_emb = bge_emb / (np.linalg.norm(bge_emb, axis=1, keepdims=True) + 1e-8)
bge_id_to_idx = {nid: i for i, nid in enumerate(bge_ids)}
print(f'BGE-M3: {bge_emb.shape}')

# 2. Node2Vec (uncorrected)
df_n2v = pd.read_parquet(N2V_PATH)
n2v_ids = df_n2v['id'].astype(int).values
n2v_emb = np.stack(df_n2v['embedding'].values)
n2v_id_to_idx = {nid: i for i, nid in enumerate(n2v_ids)}
print(f'Node2Vec: {n2v_emb.shape}')

# 3. GCN
with open(os.path.join(GCN_CACHE, 'gcn_only_results.pkl'), 'rb') as f:
    gcn_data = pickle.load(f)
gcn_node_to_idx = gcn_data['node_to_idx']
gcn_emb = gcn_data['final_embeddings']
print(f'GCN: {gcn_emb.shape}')

# 4. GraphSAGE
with open(os.path.join(SAGE_CACHE, 'graphsage_results.pkl'), 'rb') as f:
    sage_data = pickle.load(f)
sage_node_to_idx = sage_data['node_to_idx']
sage_emb = sage_data['final_embeddings']
print(f'GraphSAGE: {sage_emb.shape}')

BGE-M3: (11701, 1024)
Node2Vec: (11701, 128)
GCN: (11701, 64)
GraphSAGE: (11701, 64)


## Step 3 - Build Combined Embeddings


In [4]:
# Degree penalty array (for use during similarity computation)
degree_penalty = np.ones(len(bge_id_to_idx))
for nid, idx in bge_id_to_idx.items():
    deg = max(degrees.get(nid, 1), 1)
    degree_penalty[idx] = np.log(deg + 1) ** alpha
print(f'Degree penalty array: min={degree_penalty.min():.2f}, max={degree_penalty.max():.2f}')

# Node2Vec corrected embeddings (not used for cosine - kept for reference)
n2v_corrected_emb = n2v_emb / degree_penalty[:, np.newaxis]
print(f'Node2Vec (corrected): {n2v_corrected_emb.shape}')

# Combined embeddings (BGE-M3 + Node2Vec) - uncorrected concatenation
combined_uncorrected = np.concatenate([bge_emb, n2v_emb], axis=1)
combined_uncorrected = combined_uncorrected / (np.linalg.norm(combined_uncorrected, axis=1, keepdims=True) + 1e-8)
print(f'Combined (uncorrected): {combined_uncorrected.shape}')

# Combined (corrected) - concatenate BGE-M3 with already degree-corrected Node2Vec
combined_corrected = np.concatenate([bge_emb, n2v_corrected_emb], axis=1)
combined_corrected = combined_corrected / (np.linalg.norm(combined_corrected, axis=1, keepdims=True) + 1e-8)
print(f'Combined (corrected): {combined_corrected.shape}')


Degree penalty array: min=0.34, max=461.75
Node2Vec (corrected): (11701, 128)
Combined (uncorrected): (11701, 1152)
Combined (corrected): (11701, 1152)


## Step 4 - Compute Similarity Matrices


In [5]:
def get_sim_matrix(emb, name, cache_dir):
    """Load or compute similarity matrix with caching."""
    fpath = os.path.join(cache_dir, f'sim_{name}.pkl')
    if os.path.exists(fpath):
        print(f'  Loading {name}...')
        with open(fpath, 'rb') as f:
            return pickle.load(f)
    print(f'  Computing {name}...')
    sim = cosine_similarity(emb)
    np.fill_diagonal(sim, 0)
    with open(fpath, 'wb') as f:
        pickle.dump(sim, f)
    return sim

def get_sim_matrix_corrected(emb, penalties, name, cache_dir):
    """Compute similarity matrix WITH degree penalty applied during computation."""
    fpath = os.path.join(cache_dir, f'sim_{name}.pkl')
    if os.path.exists(fpath):
        print(f'  Loading {name}...')
        with open(fpath, 'rb') as f:
            return pickle.load(f)
    print(f'  Computing {name} (with degree correction)...')
    # Compute cosine similarity first
    sim = cosine_similarity(emb)
    np.fill_diagonal(sim, 0)
    # Apply degree penalty: sim[i,j] = sim[i,j] / (penalty[i] * penalty[j])
    # Using element-wise outer product
    penalty_matrix = np.outer(penalties, penalties)
    sim_corrected = sim / penalty_matrix
    np.fill_diagonal(sim_corrected, 0)
    with open(fpath, 'wb') as f:
        pickle.dump(sim_corrected, f)
    return sim_corrected

print('Computing/loading similarity matrices:')
sim_bge = get_sim_matrix(bge_emb, 'bge', CACHE_DIR)
sim_n2v_uncorr = get_sim_matrix(n2v_emb, 'n2v_uncorrected', CACHE_DIR)
sim_n2v_corr = get_sim_matrix_corrected(n2v_emb, degree_penalty, 'n2v_corrected', CACHE_DIR)
sim_comb_uncorr = get_sim_matrix(combined_uncorrected, 'combined_uncorrected', CACHE_DIR)
sim_comb_corr = get_sim_matrix(combined_corrected, 'combined_corrected', CACHE_DIR)
sim_gcn = get_sim_matrix(gcn_emb, 'gcn', CACHE_DIR)
sim_sage = get_sim_matrix(sage_emb, 'sage', CACHE_DIR)

print()
print('All similarity matrices ready.')


Computing/loading similarity matrices:
  Loading bge...
  Loading n2v_uncorrected...
  Loading n2v_corrected...
  Loading combined_uncorrected...
  Loading combined_corrected...
  Loading gcn...
  Loading sage...

All similarity matrices ready.


## Step 5 - Evaluation Function


In [6]:
def evaluate_model(sim_matrix, id_to_idx, E_test, k_values=[1, 10, 50, 100]):
    """
    Evaluate link prediction using precomputed similarity matrix.
    For each test edge (src, tgt), find rank of tgt among all candidates.
    """
    hits = {k: 0 for k in k_values}
    mrr = 0.0
    ndcg = 0.0
    recall_sum = 0.0
    ap_sum = 0.0
    n_evaluated = 0
    
    for src, tgt in E_test:
        if src not in id_to_idx or tgt not in id_to_idx:
            continue
        
        src_idx = id_to_idx[src]
        tgt_idx = id_to_idx[tgt]
        
        sims = sim_matrix[src_idx]
        ranked_indices = np.argsort(-sims)
        rank = np.where(ranked_indices == tgt_idx)[0][0] + 1
        
        for k in k_values:
            if rank <= k:
                hits[k] += 1
        
        mrr += 1.0 / rank
        dcg = 1.0 / np.log2(rank + 1) if rank <= max(k_values) else 0
        idcg = 1.0 / np.log2(2)
        ndcg += dcg / idcg
        
        if rank <= max(k_values):
            recall_sum += 1.0
            ap_sum += 1.0 / rank
        
        n_evaluated += 1
    
    n = max(n_evaluated, 1)
    
    return {
        **{f'Hit@{k}': hits[k] / n for k in k_values},
        'MRR': mrr / n,
        'NDCG': ndcg / n,
        'Recall': recall_sum / n,
        'MAP': ap_sum / n
    }

def rank_fusion_evaluate(sim1, id1, sim2, id2, E_test, k_values=[1, 10, 50, 100], progress_every=500):
    """
    Evaluate Reciprocal Rank Fusion (RRF) on the full shared candidate set.

    Important: candidate indices from different models are not assumed to align.
    We first align both similarity vectors onto the shared node-id set, then compute
    exact full-candidate ranks for each source node.
    """
    hits = {k: 0 for k in k_values}
    mrr = 0.0
    ndcg = 0.0
    recall_sum = 0.0
    ap_sum = 0.0
    n_evaluated = 0
    k_rrf = 60

    common_ids = sorted(set(id1.keys()) & set(id2.keys()))
    common_pos = {nid: i for i, nid in enumerate(common_ids)}
    idxs1 = np.array([id1[nid] for nid in common_ids], dtype=np.int32)
    idxs2 = np.array([id2[nid] for nid in common_ids], dtype=np.int32)

    edges_by_src = {}
    for src, tgt in E_test:
        if src not in id1 or src not in id2:
            continue
        if tgt not in common_pos:
            continue
        edges_by_src.setdefault(src, []).append(tgt)

    n_sources = len(edges_by_src)
    print(f'  Rank fusion over {n_sources} source nodes and {len(common_ids)} shared candidates...')

    for src_i, (src, targets) in enumerate(edges_by_src.items(), start=1):
        sims1 = sim1[id1[src]][idxs1]
        sims2 = sim2[id2[src]][idxs2]

        ranked1 = np.argsort(-sims1)
        ranked2 = np.argsort(-sims2)

        rankpos1 = np.empty(len(common_ids), dtype=np.int32)
        rankpos2 = np.empty(len(common_ids), dtype=np.int32)
        rankpos1[ranked1] = np.arange(1, len(common_ids) + 1)
        rankpos2[ranked2] = np.arange(1, len(common_ids) + 1)

        rrf_scores = 1.0 / (k_rrf + rankpos1) + 1.0 / (k_rrf + rankpos2)
        final_ranking = np.argsort(-rrf_scores)
        final_rankpos = np.empty(len(common_ids), dtype=np.int32)
        final_rankpos[final_ranking] = np.arange(1, len(common_ids) + 1)

        for tgt in targets:
            final_rank = final_rankpos[common_pos[tgt]]

            for k in k_values:
                if final_rank <= k:
                    hits[k] += 1

            mrr += 1.0 / final_rank
            dcg = 1.0 / np.log2(final_rank + 1) if final_rank <= max(k_values) else 0
            idcg = 1.0 / np.log2(2)
            ndcg += dcg / idcg

            if final_rank <= max(k_values):
                recall_sum += 1.0
                ap_sum += 1.0 / final_rank

            n_evaluated += 1

        if progress_every and src_i % progress_every == 0:
            print(f'    processed {src_i}/{n_sources} sources')

    n = max(n_evaluated, 1)

    return {
        **{f'Hit@{k}': hits[k] / n for k in k_values},
        'MRR': mrr / n,
        'NDCG': ndcg / n,
        'Recall': recall_sum / n,
        'MAP': ap_sum / n
    }

def rank_fusion_evaluate_fast(sim1, id1, sim2, id2, E_test, k_values=[1, 10, 50, 100], top_k=500):
    """Backward-compatible wrapper that now runs the exact aligned evaluation."""
    return rank_fusion_evaluate(sim1, id1, sim2, id2, E_test, k_values=k_values)

print('Rank fusion evaluation functions ready.')


Rank fusion evaluation functions ready.


## Step 6 - Evaluate All 8 Models


In [7]:
results = {}

print('Evaluating all 8 models (with exact aligned rank fusion)...')
print('='*70)

# Model 1: BGE-M3 Only
print('[1/7] BGE-M3 Only...')
results['1. BGE-M3 Only'] = evaluate_model(sim_bge, bge_id_to_idx, E_test)
print(f"  Hit@10: {results['1. BGE-M3 Only']['Hit@10']:.4f}, MRR: {results['1. BGE-M3 Only']['MRR']:.4f}")

# Model 2: Node2Vec Only (corrected)
print('[2/7] Node2Vec Only (corrected)...')
results['2. Node2Vec (corrected)'] = evaluate_model(sim_n2v_corr, n2v_id_to_idx, E_test)
print(f"  Hit@10: {results['2. Node2Vec (corrected)']['Hit@10']:.4f}, MRR: {results['2. Node2Vec (corrected)']['MRR']:.4f}")

# Model 3: BGE-M3 + Node2Vec (corrected)
print('[3/7] BGE-M3 + Node2Vec (corrected)...')
results['3. BGE-M3 + Node2Vec (corrected)'] = evaluate_model(sim_comb_corr, bge_id_to_idx, E_test)
print(f"  Hit@10: {results['3. BGE-M3 + Node2Vec (corrected)']['Hit@10']:.4f}, MRR: {results['3. BGE-M3 + Node2Vec (corrected)']['MRR']:.4f}")

# Model 4: GCN Only
print('[4/7] GCN Only...')
results['4. GCN Only'] = evaluate_model(sim_gcn, gcn_node_to_idx, E_test)
print(f"  Hit@10: {results['4. GCN Only']['Hit@10']:.4f}, MRR: {results['4. GCN Only']['MRR']:.4f}")

# Model 5: Node2Vec Only (uncorrected)
print('[5/7] Node2Vec Only (uncorrected)...')
results['5. Node2Vec (uncorrected)'] = evaluate_model(sim_n2v_uncorr, n2v_id_to_idx, E_test)
print(f"  Hit@10: {results['5. Node2Vec (uncorrected)']['Hit@10']:.4f}, MRR: {results['5. Node2Vec (uncorrected)']['MRR']:.4f}")

# Model 6: BGE-M3 + Node2Vec (uncorrected)
print('[6/7] BGE-M3 + Node2Vec (uncorrected)...')
results['6. BGE-M3 + Node2Vec (uncorrected)'] = evaluate_model(sim_comb_uncorr, bge_id_to_idx, E_test)
print(f"  Hit@10: {results['6. BGE-M3 + Node2Vec (uncorrected)']['Hit@10']:.4f}, MRR: {results['6. BGE-M3 + Node2Vec (uncorrected)']['MRR']:.4f}")

# Model 7: GCN + BGE-M3 Combined (exact aligned rank fusion)
print('[7/8] GCN + BGE-M3 Combined (exact rank fusion)...')
results['7. GCN + BGE-M3 (rank fusion)'] = rank_fusion_evaluate(sim_gcn, gcn_node_to_idx, sim_bge, bge_id_to_idx, E_test)
print(f"  Hit@10: {results['7. GCN + BGE-M3 (rank fusion)']['Hit@10']:.4f}, MRR: {results['7. GCN + BGE-M3 (rank fusion)']['MRR']:.4f}")

# Model 8: GraphSAGE Only
print('[8/8] GraphSAGE Only...')
results['8. GraphSAGE Only'] = evaluate_model(sim_sage, sage_node_to_idx, E_test)
print(f"  Hit@10: {results['8. GraphSAGE Only']['Hit@10']:.4f}, MRR: {results['8. GraphSAGE Only']['MRR']:.4f}")

print('='*70)
print('Evaluation complete!')


Evaluating all 8 models (with exact aligned rank fusion)...
[1/7] BGE-M3 Only...
  Hit@10: 0.1021, MRR: 0.0507
[2/7] Node2Vec Only (corrected)...
  Hit@10: 0.0795, MRR: 0.0351
[3/7] BGE-M3 + Node2Vec (corrected)...
  Hit@10: 0.0982, MRR: 0.0471
[4/7] GCN Only...
  Hit@10: 0.0620, MRR: 0.0302
[5/7] Node2Vec Only (uncorrected)...
  Hit@10: 0.1606, MRR: 0.0751
[6/7] BGE-M3 + Node2Vec (uncorrected)...
  Hit@10: 0.0024, MRR: 0.0019
[7/8] GCN + BGE-M3 Combined (exact rank fusion)...
  Rank fusion over 11256 source nodes and 11701 shared candidates...
    processed 500/11256 sources
    processed 1000/11256 sources
    processed 1500/11256 sources
    processed 2000/11256 sources
    processed 2500/11256 sources
    processed 3000/11256 sources
    processed 3500/11256 sources
    processed 4000/11256 sources
    processed 4500/11256 sources
    processed 5000/11256 sources
    processed 5500/11256 sources
    processed 6000/11256 sources
    processed 6500/11256 sources
    processed 7000/11

## Step 7 - Results Summary


In [8]:
# Create results DataFrame
df_results = pd.DataFrame(results).T

print('='*90)
print('8-MODEL COMPARISON')
print('='*90)
print(df_results.round(4).to_string())
print('='*90)

# Save results
df_results.to_csv(os.path.join(CACHE_DIR, '7_model_comparison.csv'))
with open(os.path.join(CACHE_DIR, '7_model_comparison.pkl'), 'wb') as f:
    pickle.dump(results, f)

print(f'\nResults saved to {CACHE_DIR}')

8-MODEL COMPARISON
                                     Hit@1  Hit@10  Hit@50  Hit@100     MRR    NDCG  Recall     MAP
1. BGE-M3 Only                      0.0215  0.1021  0.2436   0.3308  0.0507  0.1005  0.3308  0.0492
2. Node2Vec (corrected)             0.0157  0.0795  0.1185   0.1298  0.0351  0.0539  0.1298  0.0342
3. BGE-M3 + Node2Vec (corrected)    0.0181  0.0982  0.2465   0.3383  0.0471  0.0987  0.3383  0.0456
4. GCN Only                         0.0076  0.0620  0.2287   0.3470  0.0302  0.0842  0.3470  0.0285
5. Node2Vec (uncorrected)           0.0263  0.1606  0.4780   0.6433  0.0751  0.1766  0.6433  0.0738
6. BGE-M3 + Node2Vec (uncorrected)  0.0006  0.0024  0.0089   0.0167  0.0019  0.0039  0.0167  0.0014
7. GCN + BGE-M3 (rank fusion)       0.0153  0.1104  0.3146   0.4423  0.0495  0.1187  0.4423  0.0478
8. GraphSAGE Only                   0.0019  0.0143  0.0572   0.1033  0.0087  0.0235  0.1033  0.0072

Results saved to C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cac